# Predicting the next turn's speech features: RNN vs transformer, with and without words

**Research question.** Can a transformer predict the speech features of the upcoming turn in a conversation better than a recurrent network?

**Follow-up question.** Do the speech features themselves carry information about the next turn, or do the words of the conversation already explain it?

## Setup

- **Data:** the Fisher corpus of telephone conversations, preprocessed into one row per turn. The file is licensed to the institution, so it lives in `data/fisher/`, which is git-ignored. It is never committed.
- **Task:** for every conversation, use turns 1..t to predict the 7 speech features of turn t+1:

| feature | what it measures |
|---|---|
| `pitch_mean_log` | how high the voice is (log₁₀ Hz) |
| `pitch_std_log` | how much the pitch varies within the turn |
| `intensity_mean` | loudness (dB) |
| `jitter` | cycle-to-cycle variation in pitch |
| `shimmer` | cycle-to-cycle variation in loudness |
| `nhr` | noise-to-harmonics ratio (breathiness, hoarseness) |
| `rate_syl_sec` | speaking rate (syllables per second) |

## Experiments

| version | input for each past turn | question it answers |
|---|---|---|
| 1. speech features | the 7 features (+ missing flags, speaker) | RNN vs transformer |
| 2. words + speech features | the features plus attention over the turn's words | does adding words help? |
| 3. words only | attention over the turn's words (+ speaker) | do the features add anything beyond the words? |

If version 2 beats version 3, the speech features carry predictive information that the words don't. If they are about equal, the words already explain it.

## Settings carried over from earlier work, and what changed

| setting | used here? | why |
|---|---|---|
| the 7 features, split by `conv_id`, zero-padded start turn | yes | the task definition |
| transformer vs recurrent network | yes | the research question |
| **bidirectional RNN** | **no** | A bidirectional model also reads the conversation backwards, so its output at turn t would already contain turn t+1, the turn it is supposed to predict. It worked for IMDB because the whole review is available before classifying it. Here we predict the future, so every model must be *causal*. Section 7 tests this. |
| 25 epochs | replaced | The IMDB models overfit after 5–11 epochs. Here training stops once validation loss hasn't improved for 3 epochs (at most 30), and the best epoch is chosen by validation **loss**. That fixes the flaw noted in the IMDB notebook, where it was chosen by accuracy. |
| learning rate 3e-4 | tried, alongside 1e-3 | Each model is trained with both, and the one with the lower validation loss is kept. Transformers often need a lower learning rate than GRUs, so one shared value could favor one architecture. |
| half precision | yes, on the NVIDIA GPU | bf16 autocast. On the Mac CPU it was slower, so it is only turned on when a CUDA GPU is present. Section 8 measures the speed-up. |

## 1. Setup

In [ ]:
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from fisher_data import (FEATURES, N_FEAT, PAD_ID, UNK_ID, ConversationDataset, build_conversations,
                         build_vocab, collate_conversations, load_turns, split_conversations)
from imdb_data import BucketBatchSampler

SMOKE_TEST = os.environ.get("FISHER_SMOKE") == "1"  # small run used to check the notebook end to end
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# CUDA when available (the NVIDIA laptop), otherwise CPU. Not MPS: on this Mac the MPS backend leaked
# memory on GRUs and on transformers with varying sequence lengths (see the IMDB experiment log).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Half precision (bf16 autocast) only on an NVIDIA GPU: on the Mac CPU it made training slower (IMDB notebook).
USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "",
      "| bf16:", USE_BF16, "| torch", torch.__version__, "| smoke test:", SMOKE_TEST)

## 2. Load and check the data

The data is checked up front, so the rest of the notebook can rely on these facts instead of guarding against problems in the model code.

In [ ]:
turns = load_turns("data/fisher/fisher_data_simple.csv")
print(f"{len(turns):,} turns in {turns.conv_id.nunique():,} conversations")
turns.head()

In [ ]:
by_conv = turns.groupby("conv_id")["turn_id"]
turn_counts = by_conv.size()
assert (by_conv.min() == 1).all(), "every conversation starts at turn 1"
assert ((by_conv.max() - by_conv.min() + 1) == turn_counts).all(), "no gaps in turn_id"
assert not turns.duplicated(["conv_id", "turn_id"]).any(), "no duplicate turns"
assert set(turns.side) == {"A", "B"}

print("turns per conversation:", turn_counts.describe().round(1).to_dict())
same_speaker = (turns.side == turns.groupby("conv_id").side.shift()).mean()
print(f"turns spoken by the same person as the previous turn: {same_speaker:.1%}")

**Speakers don't strictly alternate.** About a third of turns come from the same person as the turn before. So each turn's speaker (A or B) is given to the models as a one-hot input. It isn't a speech feature, and it is part of every version.

### Missing values

Some turns are missing features. `load_turns` already treats a speaking rate of exactly 0 as missing. Those turns have no recognized words, so the rate is undefined rather than zero.

In [ ]:
missing = turns[FEATURES].isna()
summary = pd.DataFrame({
    "missing turns": missing.sum(),
    "missing %": (100 * missing.mean()).round(2),
    "mean": turns[FEATURES].mean().round(4),
    "std": turns[FEATURES].std().round(4),
})
display(summary)
no_text = turns.text.isna()
print(f"turns missing at least one feature: {missing.any(axis=1).sum():,} ({missing.any(axis=1).mean():.1%})")
print(f"turns missing all 7 features:       {missing.all(axis=1).sum():,}  (of these, {no_text[missing.all(axis=1)].mean():.0%} have no text)")
print(f"turns missing only the rate:        {(missing.rate_syl_sec & ~missing.drop(columns='rate_syl_sec').any(axis=1)).sum():,}")
print(f"conversations with a missing value: {turns.loc[missing.any(axis=1), 'conv_id'].nunique() / turns.conv_id.nunique():.0%}")

There are two kinds of missing turn:

1. **Nothing measurable** (all 7 features missing, no text): most likely laughter, a cough, noise, or a sound too short to measure.
2. **Voice measured, no words recognized** (only the rate missing): e.g. "mm-hmm", a sigh, or a filler sound. Pitch and loudness are real, but syllables per second can't be computed.

**How missing values are handled.** We keep every turn and tell the model honestly what is missing.

The alternatives are worse:
- Deleting the missing turns would make turn 6 look like it directly follows turn 4, breaking the conversation's order.
- Deleting conversations with any missing value would remove most of the dataset (see the last line above).

What we do instead:

- **As input:** a missing feature is set to the training average. That is 0 after standardizing (next section). A *missing flag* for each feature (1 = not measured) is added, so the model can tell a real average value (value 0, flag 0) from a placeholder (value 0, flag 1). The flags can also carry signal, e.g. "after laughter, the next turn tends to be louder".
- **As a target:** missing values are left out of the loss and the evaluation. The model is only ever scored on values that were actually measured. If only the rate is missing, the other 6 features are still scored.

This is done once, in the data preparation (`fisher_data.py`). The models need no special cases.

Example: a turn where someone says "mm-hmm" (voice measured, rate missing):

| | pitch | pitch var. | loudness | jitter | shimmer | nhr | rate |
|---|---|---|---|---|---|---|---|
| raw | 2.30 (200 Hz) | 0.08 | 60.0 dB | 0.010 | 0.110 | 0.20 | missing |
| value given to the model | +0.60 | −0.31 | −0.35 | −0.23 | −0.22 | −0.14 | **0.00** |
| missing flag | 0 | 0 | 0 | 0 | 0 | 0 | **1** |

## 3. Split by conversation and standardize

- **Split by `conv_id`:** 80% train, 10% validation, 10% test. A conversation is never split across sets. Otherwise the model would see part of a test conversation during training.
- **Standardize** each feature: (value − mean) ÷ std. The raw scales differ by a factor of thousands (loudness ≈ 62, jitter ≈ 0.01). After standardizing, every feature is centered on 0 with spread 1, so no feature dominates the loss just because its numbers are bigger. The mean and std come **from the training conversations only**, so nothing about the validation or test sets leaks into training.
- **Error unit:** all errors below are mean squared error in standardized units. Always predicting the training mean scores about 1.0, which makes it easy to read: 0.6 means the model removes 40% of the error of that trivial guess.

In [ ]:
train_ids, val_ids, test_ids = split_conversations(turns.conv_id.unique(), seed=SEED)
if SMOKE_TEST:
    train_ids, val_ids, test_ids = (set(sorted(ids)[:n]) for ids, n in [(train_ids, 400), (val_ids, 100), (test_ids, 100)])
assert not (train_ids & val_ids) and not (train_ids & test_ids) and not (val_ids & test_ids)

train_turns = turns[turns.conv_id.isin(train_ids)]
feature_mean = train_turns[FEATURES].mean().to_numpy(np.float32)
feature_std = train_turns[FEATURES].std().to_numpy(np.float32)
print(f"conversations: train {len(train_ids):,} | val {len(val_ids):,} | test {len(test_ids):,}")
pd.DataFrame({"train mean": feature_mean, "train std": feature_std}, index=FEATURES).round(4)

## 4. Words

Version 2 and 3 also see the words of each past turn. The transcripts are lowercase and split on spaces. The average turn has about 10 words, and the longest has 114.

- **Vocabulary** from the training conversations only.
- Words seen fewer than 5 times become a shared "unknown word" token. Very rare words can't be learned from a handful of examples. The turn and all its other words are kept.
- Turns with no transcript have no words. They still have their speech features and missing flags.

In [ ]:
MIN_WORD_COUNT = 5
vocab = build_vocab(train_turns.text.dropna(), MIN_WORD_COUNT)
vocab_size = len(vocab) + 2
train_words = train_turns.text.dropna().str.split().explode()
print(f"vocabulary: {vocab_size:,} entries (incl. padding and unknown)")
print(f"share of training words that are in the vocabulary: {train_words.isin(set(vocab)).mean():.2%}")

## 5. Building the sequences

**Start turn.** Following the suggestion to zero-pad the start of each conversation, a turn 0 is added at the beginning, so the first real turn can be predicted too. Its features are 0, all 7 missing flags are 1 ("nothing measured yet"), its speaker is neither A nor B, and it has no words.

**Shift.** For a conversation with T turns, the input is turns 0..T−1 and the target is turns 1..T. Position t of the output predicts turn t+1 using only turns 0..t. Each model is *causal*: its output at position t may not depend on any later input. This is tested in section 7.

Each input turn has 16 numbers: 7 standardized features, 7 missing flags, and 2 for the speaker.

In [ ]:
splits = {name: build_conversations(turns, ids, feature_mean, feature_std, vocab)
          for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}

# Alignment check: the input at position t+1 is exactly the target at position t.
conv = splits["train"][0]
assert torch.equal(conv.inputs[1:, :N_FEAT], conv.targets[:-1])
assert torch.equal(conv.inputs[0, :N_FEAT], torch.zeros(N_FEAT)) and conv.inputs[0, N_FEAT:2 * N_FEAT].all()
assert conv.turn_lengths.sum() == len(conv.tokens)

# Standardization check on the training targets that were actually measured.
targets = torch.cat([c.targets for c in splits["train"]])
observed = torch.cat([c.observed for c in splits["train"]])
observed_mean = (targets * observed).sum(0) / observed.sum(0)
print("train target mean per feature (should be ~0):", observed_mean.numpy().round(3))
print("turns per split:", {k: sum(len(c) for c in v) for k, v in splits.items()})

In [ ]:
BATCH_SIZE = 32  # conversations per batch (about 6,000 turns)


def make_loader(name, shuffle):
    conversations = splits[name]
    sampler = BucketBatchSampler([len(c) for c in conversations], BATCH_SIZE, shuffle=shuffle,
                                 drop_last=False, seed=SEED)
    return DataLoader(ConversationDataset(conversations), batch_sampler=sampler,
                      collate_fn=collate_conversations)


loaders = {"train": make_loader("train", True), "val": make_loader("val", False), "test": make_loader("test", False)}
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})

Batches group conversations of similar length (the length-bucketing sampler from the IMDB notebook), so little compute goes to padding. Padding sits at the end of each conversation. Because every model is causal, real positions never look at the padding after them, and padded targets are excluded from the loss by the `observed` mask.

## 6. Models

Every model has the same three parts:

1. **Turn encoder:** turns each past turn into one vector of size 128.
2. **Sequence model:** reads the turn vectors in order (GRU, GRU + attention, or transformer).
3. **Output layer:** 7 numbers, the predicted features of the next turn.

### Turn encoder and attention over words

Depending on the version, the turn vector is built from:
- the speaker (always)
- the 7 features and 7 missing flags (versions 1 and 2)
- a summary of the turn's words (versions 2 and 3).

**Attention over words.** Each word gets a learned embedding (64 numbers) and an attention score $s_i = v^\top \tanh(W e_i)$, the additive scoring from the attention notebook. The scores are softmaxed *within the turn*, and the turn's word vector is the weighted average of its embeddings. The model learns which words in a turn matter for predicting how the next turn will sound. A turn with no words gets a zero vector.

All words of a batch are kept in one flat list with the index of the turn they belong to, instead of padding every turn to 114 words.

In [ ]:
class WordAttentionPool(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=PAD_ID)
        self.W = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, 1, bias=False)

    def forward(self, tokens, token_turn, n_turns):
        embedded = self.embedding(tokens)                                  # (words, dim)
        scores = self.v(torch.tanh(self.W(embedded))).squeeze(-1)          # (words,)
        # Softmax within each turn: subtract the turn's max score for numerical stability.
        turn_max = torch.full((n_turns,), -math.inf, device=scores.device)
        turn_max = turn_max.scatter_reduce(0, token_turn, scores.detach(), "amax")
        weights = torch.exp(scores - turn_max[token_turn])
        totals = torch.zeros(n_turns, device=scores.device).index_add(0, token_turn, weights)
        weights = weights / totals[token_turn]
        pooled = torch.zeros(n_turns, embedded.shape[1], device=scores.device)
        pooled = pooled.index_add(0, token_turn, weights.unsqueeze(-1) * embedded)
        return pooled, weights


class TurnEncoder(nn.Module):
    def __init__(self, use_features, use_words, model_dim=128, word_dim=64):
        super().__init__()
        self.use_features, self.use_words = use_features, use_words
        in_dim = 2 + (2 * N_FEAT if use_features else 0) + (word_dim if use_words else 0)
        if use_words:
            self.words = WordAttentionPool(vocab_size, word_dim)
        self.proj = nn.Linear(in_dim, model_dim)

    def forward(self, batch):
        x = batch["inputs"]
        size, length, _ = x.shape
        parts = [x[..., 2 * N_FEAT:]]  # speaker
        if self.use_features:
            parts.append(x[..., :2 * N_FEAT])  # features and missing flags
        if self.use_words:
            pooled, _ = self.words(batch["tokens"], batch["token_turn"], size * length)
            parts.append(pooled.view(size, length, -1))
        return self.proj(torch.cat(parts, dim=-1))

### Sequence models

- **GRU:** one layer, hidden size 128, reading turn by turn. Padding isn't packed: padding comes after the real turns, and a one-direction GRU never looks ahead, so outputs at real positions are unchanged.
- **GRU + attention:** at each turn t, the GRU state $h_t$ attends over all GRU states so far ($h_0..h_t$), and the output uses both $h_t$ and the attention summary.
  - Scores are multiplicative, $h_t^\top W h_i / \sqrt{128}$, rather than additive.
  - Additive attention would build a vector for every (t, i) pair: 32 conversations × 457 × 457 pairs × 128 numbers is several GB. Multiplicative needs one number per pair.
  - A causal mask hides future turns.
- **Causal transformer:**
  - 2 pre-LayerNorm layers (the variant discussed in the pre-norm reading), 4 heads, model size 128, feed-forward size 256, dropout 0.1, sinusoidal positions.
  - The causal mask stops turn t from attending to later turns.
  - It uses PyTorch's `nn.TransformerEncoderLayer`. The IMDB notebook built the same block from scratch and checked it against PyTorch's attention.

In [ ]:
def causal_mask(length, device):
    # True above the diagonal = "may not attend" (future positions).
    return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)


class GRUModel(nn.Module):
    def __init__(self, encoder, hidden=128, attention=False, dropout=0.1):
        super().__init__()
        self.encoder, self.attention = encoder, attention
        self.gru = nn.GRU(encoder.proj.out_features, hidden, batch_first=True)
        if attention:
            self.W = nn.Linear(hidden, hidden, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden if attention else hidden, N_FEAT)

    def forward(self, batch):
        h, _ = self.gru(self.encoder(batch))                           # (batch, length, hidden)
        if self.attention:
            scores = h @ self.W(h).transpose(1, 2) / math.sqrt(h.shape[-1])  # [t, i] = h_t . W h_i
            scores = scores.masked_fill(causal_mask(h.shape[1], h.device), -math.inf)
            context = torch.softmax(scores, dim=-1) @ h
            h = torch.cat([h, context], dim=-1)
        return self.head(self.dropout(h))


class CausalTransformer(nn.Module):
    def __init__(self, encoder, heads=4, layers=2, ff_dim=256, dropout=0.1, max_len=1024):
        super().__init__()
        self.encoder = encoder
        dim = encoder.proj.out_features
        layer = nn.TransformerEncoderLayer(dim, heads, ff_dim, dropout, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(dim, N_FEAT)
        position = torch.arange(max_len).unsqueeze(1)
        freq = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
        pe = torch.zeros(max_len, dim)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * freq), torch.cos(position * freq)
        self.register_buffer("positions", pe)

    def forward(self, batch):
        x = self.encoder(batch)
        length = x.shape[1]
        x = self.dropout(x + self.positions[:length])
        h = self.blocks(x, mask=causal_mask(length, x.device), is_causal=True)
        return self.head(self.norm(h))


VERSIONS = {"speech features": (True, False), "words + speech features": (True, True), "words only": (False, True)}
ARCHITECTURES = {
    "GRU": lambda enc: GRUModel(enc),
    "GRU + attention": lambda enc: GRUModel(enc, attention=True),
    "transformer": lambda enc: CausalTransformer(enc),
}


def make_model(version, architecture):
    use_features, use_words = VERSIONS[version]
    return ARCHITECTURES[architecture](TurnEncoder(use_features, use_words)).to(device)

## 7. Sanity checks

1. **Causality:** change everything after position k (features and words) and check that the outputs up to position k don't change. If this failed, a model could "see" the turn it is predicting.
2. **Attention over words:** the weights within each turn that has words sum to 1.

In [ ]:
def to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}


def perturb_after(batch, k):
    changed = {key: value.clone() for key, value in batch.items()}
    changed["inputs"][:, k + 1:] = torch.randn_like(changed["inputs"][:, k + 1:])
    length = batch["inputs"].shape[1]
    later_words = (batch["token_turn"] % length) > k
    changed["tokens"][later_words] = torch.randint(2, vocab_size, (int(later_words.sum()),), device=device)
    return changed


check_batch = to_device(next(iter(loaders["val"])))
k = 20
for version in VERSIONS:
    for architecture in ARCHITECTURES:
        model = make_model(version, architecture).eval()
        with torch.no_grad():
            original = model(check_batch)
            changed = model(perturb_after(check_batch, k))
        assert torch.allclose(original[:, :k + 1], changed[:, :k + 1], atol=1e-5), (version, architecture)
        assert (original[:, k + 1:] - changed[:, k + 1:]).abs().max() > 1e-3, "later outputs should change"
print("causality check passed for all 9 models: outputs up to turn k ignore everything after it")

pool = make_model("words only", "GRU").encoder.words
size, length = check_batch["inputs"].shape[:2]
with torch.no_grad():
    _, weights = pool(check_batch["tokens"], check_batch["token_turn"], size * length)
sums = torch.zeros(size * length, device=device).index_add(0, check_batch["token_turn"], weights)
has_words = torch.bincount(check_batch["token_turn"], minlength=size * length) > 0
assert torch.allclose(sums[has_words], torch.ones_like(sums[has_words]), atol=1e-5)
print("word attention weights sum to 1 within every turn that has words")

## 8. Training

- **Loss:** mean squared error over the measured target values only (the `observed` mask).
- **Optimizer:** Adam with gradient clipping at 1.0. Each model is trained at learning rates 1e-3 and 3e-4, and the one with the lower validation loss is kept. The test set is only used for the kept one.
- **Precision:** on the GPU, the forward pass runs in bf16 (autocast), while the loss and the weights stay in full precision.
- **Model selection:** the best epoch is chosen by **validation loss**, not accuracy-style metrics. The IMDB experiment showed why: picking by accuracy kept overconfident late epochs. Training stops early when validation loss hasn't improved for 3 epochs (at most 30 epochs).

In [ ]:
LEARNING_RATES = [1e-3, 3e-4]
MAX_EPOCHS = 2 if SMOKE_TEST else 30
PATIENCE = 3


def autocast():
    return torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16)


def evaluate(predict, loader):
    # Returns per-feature sum of squared errors and number of measured values.
    sse = torch.zeros(N_FEAT)
    count = torch.zeros(N_FEAT)
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch)
            observed = batch["observed"]
            with autocast():
                prediction = predict(batch)
            error = (prediction.float() - batch["targets"]) ** 2 * observed
            sse += error.sum(dim=(0, 1)).cpu()
            count += observed.sum(dim=(0, 1)).cpu()
    return sse, count


def mse(sse, count):
    return float(sse.sum() / count.sum())


def train_model(version, architecture, lr):
    torch.manual_seed(SEED)
    model = make_model(version, architecture)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best = [], (math.inf, None, 0)
    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        model.train()
        train_sse, train_count = 0.0, 0.0
        for batch in loaders["train"]:
            batch = to_device(batch)
            observed = batch["observed"]
            with autocast():
                prediction = model(batch)
            squared = (prediction.float() - batch["targets"]) ** 2 * observed  # loss in full precision
            loss = squared.sum() / observed.sum()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_sse += squared.detach().sum().item()
            train_count += observed.sum().item()
        model.eval()
        val_loss = mse(*evaluate(model, loaders["val"]))
        history.append({"epoch": epoch, "train": train_sse / train_count, "val": val_loss, "seconds": time.time() - start})
        print(f"  epoch {epoch:2d}  train {history[-1]['train']:.4f}  val {val_loss:.4f}  ({history[-1]['seconds']:.0f}s)")
        if val_loss < best[0]:
            best = (val_loss, {k: v.detach().clone() for k, v in model.state_dict().items()}, epoch)
        elif epoch - best[2] >= PATIENCE:
            break
    model.load_state_dict(best[1])
    model.eval()
    return model, pd.DataFrame(history), best[2]

### Half precision: does it speed up training?

Time 30 training steps per architecture (version 1 inputs, a typical batch), in full precision (fp32) and in bf16. This only runs on the NVIDIA GPU. On the Mac CPU, bf16 was slower (IMDB notebook).

In [ ]:
def time_steps(architecture, use_bf16, steps=30):
    torch.manual_seed(SEED)
    model = make_model("speech features", architecture).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    batch = to_device(next(iter(loaders["train"])))
    for i in range(steps + 5):  # first 5 steps are warm-up
        if i == 5:
            torch.cuda.synchronize()
            start = time.time()
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            prediction = model(batch)
        loss = (((prediction.float() - batch["targets"]) ** 2) * batch["observed"]).sum() / batch["observed"].sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    return 1000 * (time.time() - start) / steps


if device.type == "cuda":
    timing = pd.DataFrame([{"architecture": a, "fp32 ms/step": time_steps(a, False), "bf16 ms/step": time_steps(a, True)}
                           for a in ARCHITECTURES])
    timing["speed-up"] = (timing["fp32 ms/step"] / timing["bf16 ms/step"]).round(2)
    display(timing.round(1))
else:
    print("No CUDA GPU: skipped. Training runs in fp32.")

## 9. Baselines

A model is only useful if it beats simple rules. There are two:

- **Training mean:** always predict the average turn. This scores about 1.0 by construction.
- **Previous turn:** predict that the next turn sounds exactly like the last one. Before the first turn, and for features that weren't measured, it falls back to the average.

In [ ]:
results = []
baselines = {
    "training mean": lambda batch: torch.zeros_like(batch["targets"]),
    "previous turn": lambda batch: batch["inputs"][..., :N_FEAT],
}
for name, predict in baselines.items():
    val_sse, val_count = evaluate(predict, loaders["val"])
    test_sse, test_count = evaluate(predict, loaders["test"])
    results.append({"version": "baseline", "model": name, "params": 0, "lr": None, "best epoch": None,
                    "val MSE": mse(val_sse, val_count), "test MSE": mse(test_sse, test_count),
                    **dict(zip(FEATURES, (test_sse / test_count).numpy()))})
pd.DataFrame(results)[["model", "val MSE", "test MSE"]].round(4)

## 10. Train every model

Three versions × three architectures × two learning rates. All share the same split, standardization, batches, learning rate and stopping rule.

In [ ]:
histories, models = {}, {}
for version in VERSIONS:
    for architecture in ARCHITECTURES:
        name = f"{architecture} | {version}"
        start = time.time()
        runs = {}
        for lr in LEARNING_RATES:
            print(f"{name} | lr {lr:g}")
            model, history, best_epoch = train_model(version, architecture, lr)
            runs[lr] = (mse(*evaluate(model, loaders["val"])), model, history, best_epoch)
        best_lr = min(runs, key=lambda lr: runs[lr][0])
        val_loss, model, history, best_epoch = runs[best_lr]
        test_sse, test_count = evaluate(model, loaders["test"])
        results.append({"version": version, "model": architecture,
                        "params": sum(p.numel() for p in model.parameters()), "lr": best_lr, "best epoch": best_epoch,
                        **{f"val MSE (lr {lr:g})": runs[lr][0] for lr in LEARNING_RATES},
                        "val MSE": val_loss, "test MSE": mse(test_sse, test_count),
                        "train min": (time.time() - start) / 60,
                        **dict(zip(FEATURES, (test_sse / test_count).numpy()))})
        histories[name], models[name] = history, model
        print(f"  -> kept lr {best_lr:g} (best epoch {best_epoch}), test MSE {results[-1]['test MSE']:.4f}\n")

## 11. Results

In [ ]:
table = pd.DataFrame(results)
mean_mse = table.loc[table.model == "training mean", "test MSE"].item()
table["error removed vs mean"] = (1 - table["test MSE"] / mean_mse).map("{:.1%}".format)
display(table[["version", "model", "params", "lr", "best epoch", *[f"val MSE (lr {lr:g})" for lr in LEARNING_RATES],
               "test MSE", "error removed vs mean", "train min"]].round(4))

Test MSE per feature (standardized units; lower is better):

In [ ]:
table.set_index(["version", "model"])[FEATURES].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, version in zip(axes, VERSIONS):
    for architecture in ARCHITECTURES:
        history = histories[f"{architecture} | {version}"]
        line, = ax.plot(history.epoch, history.val, label=f"{architecture} (val)")
        ax.plot(history.epoch, history.train, "--", color=line.get_color(), alpha=0.5)
    ax.set_title(version)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("MSE (dashed = train, solid = validation)")
axes[0].legend()
plt.tight_layout()
plt.show()

### Which words does the model attend to?

For the best words + speech features model, these are the highest-weighted words in a few test turns.

In [ ]:
word_models = {k: v for k, v in models.items() if k.endswith("| words + speech features")}
best_name = min(word_models, key=lambda k: table.loc[(table.version == "words + speech features")
                                                     & (table.model == k.split(" | ")[0]), "val MSE"].item())
id_to_word = {i: w for w, i in vocab.items()} | {UNK_ID: "<unk>"}
batch = to_device(next(iter(loaders["test"])))
size, length = batch["inputs"].shape[:2]
with torch.no_grad():
    _, weights = models[best_name].encoder.words(batch["tokens"], batch["token_turn"], size * length)
print(best_name)
shown = 0
for turn in range(1, length):
    selected = (batch["token_turn"] == turn).nonzero().squeeze(-1)
    if len(selected) < 6:
        continue
    words = [id_to_word[int(t)] for t in batch["tokens"][selected]]
    w = weights[selected].cpu().numpy()
    print(" ".join(f"{word}[{weight:.2f}]" if weight >= np.sort(w)[-2] else word for word, weight in zip(words, w)))
    shown += 1
    if shown == 8:
        break

## 12. What the results say

*Written after the full run; see below.*